In [ ]:
import pandas as pd
from paths import ROOT, DATA

# --- Q1: relative paths -------------------------------------------------------
# Directory constants live in src/paths.py, resolved from that module's own
# __file__ - modules have one, notebooks do not, which is why the definitions
# belong in a module. Append your own filename at the call site.
#
# No sys.path bootstrap: `pip install -e .` puts src on the import path for
# every entry point - notebook, script, pytest - from any working directory.
print(ROOT)

In [ ]:
df = pd.read_csv(DATA / "accepted_36_month_population.csv", low_memory=False)

# was: pd.read_csv(r"C:\Users\HD1225\OneDrive\Desktop\Lending-Club\src\data\...")
#
# low_memory=False: pandas otherwise infers dtypes chunk-by-chunk and warns on the
# mixed-type object columns. Cheap at 590k rows, and it makes the dtype column of
# the profile below trustworthy.
df.shape

In [ ]:
# ---- your version ------------------------------------------------------------
a = df.nunique().to_dict()
b = []
for key, value in a.items():
    if value == 0 :
        b.append({"column_name": key, "unique_values": value})

empty_columns = [i["column_name"] for i in b]

# ---- alternative: one line, vectorised ---------------------------------------
# nunique() already returns a Series indexed BY COLUMN NAME. .to_dict() throws away
# the index alignment that makes it useful, and you then rebuild it by hand:
#
#     nu = df.nunique()
#     empty_columns = nu[nu == 0].index.tolist()
#
# The list-of-dicts held nothing `nu` did not already hold - note that you flatten
# it straight back out on the next line. Two data structures and a loop to express
# one boolean mask.
#
# Speed is not the point at 151 columns. The habit is: any `for key, value in
# X.items()` over a pandas object has a vectorised form, and the vectorised form
# stays correct when you later want "nunique <= 1" instead of "== 0".
#
# Naming: `a` and `b` are unreadable in a week. When a variable resists a name it
# usually means the STEP was never named either - here it is "column profile" and
# "all-null columns", and the code reads better once you say so.

In [ ]:
# ---- your version ------------------------------------------------------------
for col in empty_columns:
    print(df[col].value_counts())

# value_counts() on an all-null column returns an EMPTY Series, so this prints 15
# empty frames - it cannot confirm the thing you want confirmed. The check you
# actually mean is one line, and it passes silently or fails loudly:
#
#     assert df[empty_columns].notna().sum().eq(0).all()
#
# Prefer an assert over a print for anything you are verifying rather than reading.
# A print needs a human to notice it; an assert stops the notebook.

In [ ]:
# ==============================================================================
# The tool that replaces all of the above: ONE profile frame, then query it.
# ==============================================================================
# Every Step-3 triage question is a filter on this table. Compute it once instead
# of writing a fresh loop per question - and it is the first draft of the column
# spec the step is supposed to deliver.
profile = pd.DataFrame({
    "dtype":    df.dtypes.astype(str),
    "n_null":   df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "nunique":  df.nunique(),        # <-- dropna=True by default. See two cells down.
})
profile.index.name = "column"

profile.sort_values("null_pct", ascending=False).head(20)

In [ ]:
# ---- Q2: the families your nunique()==0 check cannot see ---------------------
all_null   = profile.query("nunique == 0")                        # 15  <- you found these
constant   = profile.query("nunique == 1")                        # 11  <- MISSED
near_empty = profile.query("null_pct >= 97 and null_pct < 100")   # 37  <- MISSED

print(f"all-null   {len(all_null):3d}")
print(f"constant   {len(constant):3d}   {sorted(constant.index)}")
print(f"near-empty {len(near_empty):3d}")

# `constant` is a genuinely different family: 0% null, one repeated value, zero
# information. policy_code, pymnt_plan, disbursement_method, hardship_flag - plus
# term and out_prncp, which are constant only BECAUSE of your Step 1 and Step 2
# filters. That distinction belongs in the spec: "constant in this population" is
# a different claim from "constant".
#
# `near_empty` is the bigger miss - 37 columns, none of them empty, so no nunique
# test will ever surface them. Three sub-families, three different verdicts:
#   97.48%  open_acc_6m, il_util, all_util, ...  LC added them ~Dec 2015, at the
#           very end of the window -> vintage artefact, not a feature
#   98-99%  settlement_*, hardship_*             post-origination -> banned by 3b
#   99.96%  dti_joint, annual_inc_joint, ...     239 joint apps, see below

In [ ]:
# ==============================================================================
# Q: what use is a 99.96%-null column?  A: none - but that is NOT a null-rate rule
# ==============================================================================
# You are right that 239 non-null rows out of 589,488 cannot support anything. But
# be careful about the generalisation: high null rate is not by itself a reason to
# drop. Look at where the null rates actually fall in this population.

bands = pd.cut(profile["null_pct"],
               [-0.01, 0.001, 1, 10, 50, 97, 99.999, 100.01],
               labels=["0%", "<1%", "1-10%", "10-50%", "50-97%", "97-100%", "100%"])
print(bands.value_counts().sort_index().to_string())

print()
print(profile.query("null_pct >= 10 and null_pct < 97")
             .sort_values("null_pct", ascending=False)[["null_pct", "nunique"]]
             .to_string())

In [ ]:
# ---- read the output above carefully -----------------------------------------
# There is a clean EMPTY GAP between 87.28% (desc) and 97.48% (open_acc_6m). No
# column lands in it. So a threshold anywhere in 88-97 cuts the same set, and you
# can justify it from the data instead of picking a round number and hoping:
#
#     DROP_IF_NULL_ABOVE = 95.0   # inside a natural 10-point gap; cuts 52 columns
#
# Write the threshold down as a named constant with that one-line reason. A magic
# 0.95 buried in a filter is the thing nobody can defend in review.
#
# Now the important half. Everything in the 50-87% band is a `mths_since_*` column:
#
#     mths_since_last_record            83.47%
#     mths_since_recent_bc_dlq          75.28%
#     mths_since_last_major_derog       73.19%
#     mths_since_recent_revol_delinq    65.32%
#     mths_since_last_delinq            50.51%
#
# These are heavily null AND genuinely predictive. "Months since last public
# record" is null for 83% of borrowers because 83% of borrowers HAVE NO PUBLIC
# RECORD - that is not missing data, it is a strong signal about a good borrower.
# "Never happened" is not "unknown". Drop these on a null-rate rule and you throw
# away some of the best features in the file; median-impute them in Step 8 and you
# invent a delinquency that never occurred. They need a sentinel plus an explicit
# indicator flag.
#
# `desc` at 87.28% sits above the same line and goes anyway - free text, high
# cardinality, a text-modelling project rather than a leakage or nullity problem.
# Different reason, and the spec has to say which.
#
# So the rule is:
#   > 97%   nothing to learn from 0.04%-2.5% of rows -> drop, whatever it means
#   50-97%  nullity IS the signal -> keep, handle deliberately in Step 8
#   < 50%   keep, decide the fill strategy per column family in Step 8
#
# Null rate produces a SHORTLIST above the gap and a QUESTION below it. It is only
# a verdict at the very top, where there is not enough data to learn from at all.
#
# One thing this check still cannot see: sentinel values that are not NaN.
# dti == 999 is LC's "undefined" marker and isna() reads it as a perfectly good
# number. Same for "n/a" strings in object columns. That is Step 5, but know now
# that isna() is not the whole missingness story.

In [ ]:
# ---- the nunique() trap ------------------------------------------------------
# nunique() DROPS NaN. So "nunique == 1" means "one distinct value among the rows
# that have one", NOT "constant across the population". These four look alike
# through nunique alone and are four different situations:
profile.loc[["policy_code", "term", "verification_status_joint", "hardship_type"]]

# null_pct and nunique answer two different questions - "how much is here" and
# "how varied is what is here" - and you need them side by side to tell a real
# constant from a near-empty column that happens to be uniform.

In [ ]:
# ---- the 239 joint applications ----------------------------------------------
jt = df[df["application_type"].str.strip() == "Joint App"]
print(len(jt), jt["issue_d"].min(), jt["issue_d"].max())

# 239 loans, ALL issued 2015-10..2015-12 - entirely inside the TEST window
# (2015-04-01 onward). So dti_joint is not a free deletion: it is a column that
# exists only in test and never in train. A model cannot learn from it, and its
# presence is a perfect proxy for "late 2015".
#
# This is why 99.96% null and 100% null are worth separating. The findings doc had
# these three in the free-deletion bucket; measuring the population showed they are
# a vintage problem instead. Same drop, completely different reason - and the
# reason is the deliverable.

In [ ]:
# ---- what is still left in Step 3 --------------------------------------------
# Nulls and variance are the MECHANICAL half (3a). Still outstanding:
#
#   3a  identifiers / free text / PII : id, url, zip_code, emp_title, title, desc
#                                       (all pass the time-machine test - they go
#                                        for cardinality, PII and fair lending)
#   3b  the time-machine test         : sort every survivor by clock,
#                                       application / bureau-pull / servicing
#   3b  axis 1 audit                  : null rate by loan_status  -> leakage
#   4   axis 2                        : null rate by issue year   -> generalisation
#
# Output is a verdict + reason for all 151 columns, not a filtered df. The profile
# frame above is where those two columns get added:
#
# profile["keep"]   = ...
# profile["reason"] = ...   # leakage | empty | constant | identifier | policy
# assert set(profile.index) == set(df.columns)